# Day 2 Homework — AI Product Intelligence System
### Tasks implemented: Smart Recommendations, Unique Catalog Creation, Reverse (Text) Search

This notebook ties together the backend modules (`data_loader.py`, `clip_embeddings.py`,
`recommendation_engine.py`, `catalog_dedup.py`, `search_engine.py`, `visualization.py`)
into runnable demos for all three tasks, using CLIP image/text embeddings as the shared
representation across tasks.

**Dataset:** Fashion Product Images (Small) — https://www.kaggle.com/code/sahandakramipour/fashion-product-images-small

If running on Kaggle, attach the dataset and it will be auto-detected at
`/kaggle/input/fashion-product-images-small`.


## 0. Setup

In [ ]:
!pip install -q transformers torch --upgrade


In [ ]:
import os, sys
sys.path.append(os.path.abspath("."))  # if backend .py files are alongside this notebook

import config
print("Data dir:", config.DATA_DIR)
print("Images dir:", config.IMAGES_DIR)
print("Styles CSV:", config.STYLES_CSV)


## 1. Load the product catalog

We load `styles.csv` and resolve each product's image path, dropping any rows whose
image file is missing (the raw dataset has a few of these). For speed on a homework
scale, we work with a random sample of the catalog — set `sample_size=None` to use the
full ~44k products.

In [ ]:
from data_loader import ProductCatalog

catalog = ProductCatalog.load(sample_size=2000)  # set to None for the full dataset
print(f"Loaded {len(catalog)} products")
catalog.df.head()


## 2. Build CLIP embeddings

We embed every product **image** into a shared CLIP vector space. The same embedding
space will later let us embed a **text** query and compare it directly against image
vectors (cosine similarity) — this is what powers Task 3, and it's also used to
re-rank style compatibility in Task 1 and to detect near-duplicate photos in Task 2.

Embeddings are cached to disk (`outputs/embeddings_cache.npz`) so re-running this cell
after the first time is instant.

In [ ]:
from clip_embeddings import CLIPEmbedder, build_or_load_image_embeddings

embedder = CLIPEmbedder()  # openai/clip-vit-base-patch32 by default

product_ids = catalog.all_ids()
image_paths = catalog.df["image_path"].tolist()

embeddings = build_or_load_image_embeddings(product_ids, image_paths, embedder=embedder)
print("Embeddings shape:", embeddings.shape)


---
## Task 1: Smart Product Recommendation Engine

**Goal:** given a product, recommend things that are commonly bought *with* it
(complementary), not just things that *look* like it.

**How it works (hybrid rule + embedding approach):**
1. A hand-authored category-compatibility map encodes outfit/usage knowledge
   (e.g. `Casual Shoes -> Socks, Watches, Backpacks, Sunglasses, Belts`). This stands in
   for real co-purchase/basket data, which this dataset doesn't include — and the API
   (`fit_cooccurrence`) is designed to swap in real transaction logs later with no
   other code changes.
2. Candidates are filtered to match the seed product's **gender** (and optionally
   season).
3. Remaining candidates are **re-ranked by CLIP image-embedding similarity** to the
   seed product, which acts as a rough colour/style compatibility score — e.g.
   preferring a black watch to match black shoes over a brightly-coloured one.

Every recommendation comes with a human-readable `reason` string for explainability.

In [ ]:
from recommendation_engine import RecommendationEngine
import visualization as viz

reco_engine = RecommendationEngine(catalog, image_embeddings=embeddings, embedding_ids=product_ids)

# Pick a demo seed product — first available "Casual Shoes" / footwear-ish item
shoe_candidates = catalog.filter(articleType="Casual Shoes")
seed_id = shoe_candidates.iloc[0]["id"] if not shoe_candidates.empty else product_ids[0]

print(reco_engine.explain(seed_id, top_k=5))


In [ ]:
recs = reco_engine.recommend(seed_id, top_k=5)
fig = viz.plot_recommendations(catalog, seed_id, recs)
fig


---
## Task 2: Unique Product Catalog Creation

**Goal:** large marketplaces end up with many near-duplicate listings (the same
physical product re-photographed/re-uploaded by different sellers). Build a clean
catalog that keeps only one entry per actual product.

**How it works:**
1. Compute pairwise cosine similarity between CLIP image embeddings.
2. Connect any two products whose similarity exceeds a threshold (default `0.97`)
   into the same group, using **connected components / Union-Find** — this correctly
   merges duplicate *chains* (A~B~C) rather than only pairwise matches.
3. Within each group, keep the product with the most descriptive
   (longest) `productDisplayName` as the **representative**; everything else in the
   group is reported as a duplicate.

In [ ]:
from catalog_dedup import build_clean_catalog

dedup_result = build_clean_catalog(catalog, embeddings, product_ids, threshold=0.97)

print(f"Original products:   {dedup_result.n_original}")
print(f"Unique products:     {dedup_result.n_unique}")
print(f"Duplicate groups found: {dedup_result.n_duplicate_groups}")
dedup_result.duplicate_map.head(10)


In [ ]:
# Visualize one detected duplicate group (if any were found at this threshold)
if not dedup_result.duplicate_map.empty:
    rep = dedup_result.duplicate_map.iloc[0]["representative_id"]
    members = [rep] + dedup_result.duplicate_map[
        dedup_result.duplicate_map["representative_id"] == rep
    ]["duplicate_id"].tolist()
    fig = viz.plot_dedup_group(catalog, members, rep)
    fig
else:
    print("No near-duplicates found at this threshold/sample — try a larger sample "
          "or lower the threshold slightly (e.g. 0.95) to see example groups.")


In [ ]:
# Save the clean catalog + duplicate map
dedup_result.clean_catalog.to_csv(config.CLEAN_CATALOG_CSV, index=False)
dedup_result.duplicate_map.to_csv(config.DUPLICATE_MAP_CSV, index=False)
print("Saved:", config.CLEAN_CATALOG_CSV, "and", config.DUPLICATE_MAP_CSV)


---
## Task 3: Reverse Product Search (text → image)

**Goal:** let a user type a description (e.g. *"blue casual shirt"*) instead of
uploading a photo, and retrieve the best-matching product images.

**How it works:** CLIP embeds images and text into the **same** vector space, so we
simply embed the query text and rank all product image embeddings by cosine
similarity to it — no separate text index or keyword matching needed.

In [ ]:
from search_engine import ReverseSearchEngine

search_engine = ReverseSearchEngine(catalog, embeddings, product_ids, embedder=embedder)

query = "blue casual shirt"
print(search_engine.pretty_print(query, top_k=5))


In [ ]:
results = search_engine.search(query, top_k=5)
fig = viz.plot_search_results(query, results)
fig


In [ ]:
# Try your own query
my_query = "red sports shoes"  
print(search_engine.pretty_print(my_query, top_k=5))
fig = viz.plot_search_results(my_query, search_engine.search(my_query, top_k=5))
fig


: 

---
## Summary

| Task | Technique | Output |
|---|---|---|
| 1. Recommendation Engine | Category-compatibility rules + gender/season filtering + CLIP style re-ranking | Top-K complementary products with explanations |
| 2. Unique Catalog Creation | CLIP image embeddings + similarity-graph connected components | Deduplicated catalog CSV + duplicate map CSV |
| 3. Reverse Product Search | Shared CLIP image/text embedding space, cosine similarity ranking | Top-K matching products for a free-text query |

All three tasks share the same embedding backbone (`clip_embeddings.py`), so embeddings
computed once are reused everywhere — image embeddings power recommendation re-ranking,
duplicate detection, *and* serve as the search index for text queries.

A FastAPI backend (`app.py`) wraps these same engines as HTTP endpoints
(`/recommend/{id}`, `/search`, `/catalog/dedup`) for use outside the notebook.